# StorageGuard experiments on a Colab GPU

This notebook runs every LLM experiment of the StorageGuard paper on one GPU:

| Stage | What | Estimated time on an A100 |
|---|---|---|
| 1-3 | check GPU, mount Drive, install | 10 min |
| 4 | **smoke test**: 12 short episodes per model (first run also downloads ~15 GB per model) | 20-30 min |
| 5 | full run: dev split, main matrix (3 models x 5 configs x 3 runs), ablations, report | 2-4 h (estimate) |
| 6-7 | show the report, download the results | 1 min |

**Before you start:** `Runtime -> Change runtime type -> A100 GPU`. For Llama (gated, approved by Meta), add a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, enable *Notebook access*).

Everything is resumable: results are mirrored to Google Drive every 5 minutes. If the session disconnects, reconnect and run cells 1, 2, 3 again, then cell 5; finished episodes are skipped.

## 1. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU")
assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU"
if "A100" not in gpu and "H100" not in gpu:
    print("WARNING: not an A100/H100. 7-8B models still fit on a 24 GB GPU (L4), but expect it to be slower.")

## 2. Settings, Google Drive and the project

Upload `storageguard_colab.zip` to `MyDrive/storageguard/` first (see COLAB.md).

In [ ]:
# ---- settings -------------------------------------------------------------
BACKEND   = "vllm"                                  # "ollama" = fallback if vLLM cannot be installed or started
MODELS    = "qwen2.5-7b,llama3.1-8b,ministral-3-8b" # scale study: see StorageGuard_Colab_Continue.ipynb, Part B
MAIN_RUNS = 3                                       # runs per (model, config) in the main experiment
DRIVE_DIR = "/content/drive/MyDrive/storageguard"   # where you uploaded storageguard_colab.zip
# ---------------------------------------------------------------------------

import os, sys, time, zipfile, pathlib, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
bundles = sorted(pathlib.Path(DRIVE_DIR).glob("storageguard_colab*.zip"), key=os.path.getmtime)
assert bundles, f"Upload storageguard_colab.zip to {DRIVE_DIR} first"
bundle = bundles[-1]                          # the newest upload, also if Drive kept an older copy
zipfile.ZipFile(bundle).extractall("/content")  # always the newest code; results are never in the zip
print("code from", bundle.name, "uploaded", time.strftime("%Y-%m-%d %H:%M", time.localtime(bundle.stat().st_mtime)))
os.chdir("/content/storageguard")
os.environ["SG_DRIVE_STATE"] = f"{DRIVE_DIR}/state"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as exc:  # secret missing or notebook access not granted
    print(f"No HF_TOKEN ({type(exc).__name__}). Qwen and Mistral will work; Llama downloads will fail.")

def run(*args):
    """Run the driver, stream its output here, and stop the notebook if it fails."""
    p = subprocess.Popen([sys.executable, "-u", "colab/sg_colab.py", *map(str, args)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed with exit code {p.returncode}; read the messages above")

def show_report():
    from IPython.display import Markdown, Image, display
    display(Markdown(pathlib.Path("results/REPORT.md").read_text()))
    for png in sorted(pathlib.Path("paper/figures").glob("*.png")):
        print(png.name)
        display(Image(filename=str(png), width=900))

def download_results():
    run("pack")   # zips results/, paper/, data/bench and configs into DRIVE_DIR
    from google.colab import files
    latest = max(pathlib.Path(DRIVE_DIR).glob("storageguard_results_*.zip"), key=os.path.getmtime)
    print("saved to Drive:", latest)
    files.download(str(latest))

print("project ready at", os.getcwd())

## 3. Install (about 5-10 minutes)

Creates two isolated environments: one for the experiment harness (Python 3.11) and one for vLLM (Python 3.12), then runs the fast unit tests.

In [ ]:
run("setup", "--backend", BACKEND)

## 4. Smoke test (do not skip)

For each model: start the server, run 3 dev-split trap tasks under 4 configurations (no guard, RBAC, LLM judge, StorageGuard), and check that tool calls parse, the guard is reached, the judge is called and episodes finish. It first checks (in seconds) that your Hugging Face token can download each model. A model that fails is reported with the reason and the next one is still tested; a summary comes at the end. The first run downloads each model.

In [ ]:
run("smoke", "--backend", BACKEND, "--models", MODELS)

## 5. Full run

Primary model (Qwen2.5-7B): dev split (all 9 configs), main matrix, ablations + threshold sweep + approver sensitivity. Other models: main matrix. Then the report. Safe to re-run: finished episodes are skipped and results are restored from Drive. To add a model later (for example once its licence is approved), smoke-test it alone and run this cell with only that model: `run("all", "--backend", BACKEND, "--models", "llama3.1-8b", "--main-runs", MAIN_RUNS)`.

In [ ]:
run("all", "--backend", BACKEND, "--models", MODELS, "--main-runs", MAIN_RUNS)

## 6. Report

In [ ]:
show_report()

## 7. Download the results

In [ ]:
download_results()   # also saved to DRIVE_DIR

## Troubleshooting

| Symptom | Fix |
|---|---|
| Session disconnected | Reconnect, run cells 1, 2, 3, then 5 again. Nothing finished is lost. |
| `403` / "not in the authorized list" for Llama or Mistral | The account behind `HF_TOKEN` has no access yet. Open the model page while logged in: submit the licence form, or wait if it says "awaiting review" (Llama is approved by hand). If it says "granted", use a classic *Read* token (a fine-grained one needs "Read access to contents of all public gated repos"), update the secret and re-run cell 2. Meanwhile run the other models: set `MODELS` without it. |
| `401` for Llama or Mistral | No token or a rejected one: check the `HF_TOKEN` secret (Notebook access on) and re-run cell 2. |
| vLLM install fails or the server exits at start | Read the log tail printed above. Setup installs the pinned vLLM 0.33.0 with matching packages; keep it for the whole study. Last resort: `BACKEND = "ollama"` (results stored as `<model>-q4`, never mixed with vLLM ones). |
| Mistral-format model: `400 ... missing ... 'force_tool_names'` | mistral_common too new for vLLM 0.33.0: run cell 3 again with the current zip, or `!uv pip install --python /content/venv-vllm/bin/python "mistral-common[image]==1.12.0"`. |
| CUDA out of memory | Add `"--concurrency", 8` to the `run(...)` calls. |
| Smoke test: "never produced a valid tool call" | The tool-call parser or chat template does not match the model. Drop that model from `MODELS` and report it as a limitation, or try the Ollama backend. |
| Want to check progress | `!find results/runs -name '*.json' | wc -l` in a new cell, or look at `results/runs` in the Files pane. |